# GPU worker for photoreal splats

This notebook turns a free Colab GPU into a **worker** for your platform. It connects to the
worker gateway on your laptop, picks up every project that asked for a *photoreal Gaussian
splat*, trains it with Nerfstudio **splatfacto**, and uploads the result. The splat then
appears in the 3D Studio's **Walk** mode, with no downloading or copying by hand.

```text
Your laptop                                   Colab (this notebook)
  web platform  :8000                          ┌───────────────────────────┐
  worker gateway :8001 ◄── cloudflared tunnel ─┤ python -m queue.workers   │
  mesh worker                                  │   --pipeline splat        │
                                               │ claim → download video →  │
                                               │ COLMAP → splatfacto →     │
                                               │ upload point_cloud.ply    │
                                               └───────────────────────────┘
```

**Before you start, on your laptop:** the API, the worker gateway, and the tunnel are running
(README → *Photoreal splats on a GPU*). You need the tunnel URL and your `WORKER_TOKEN`.

**Runtime → Change runtime type → T4 GPU** (or better), then run the cells in order.

## 1. Check the GPU

In [ ]:
import subprocess, torch
assert torch.cuda.is_available(), "Runtime → Change runtime type → GPU, then run this cell again"
print("GPU:", torch.cuda.get_device_name(0), "| torch", torch.__version__)

## 2. Install COLMAP, Nerfstudio, and this project (about 5–10 minutes)

In [ ]:
!apt-get -qq update && apt-get -qq install -y colmap ffmpeg > /dev/null
!pip -q install nerfstudio
import os
REPO = "/content/ai-indoor-3d-reconstruction"
if not os.path.isdir(REPO):
    !git clone -q https://github.com/prakrity26/ai-indoor-3d-reconstruction.git {REPO}
else:
    !git -C {REPO} pull -q
# --no-deps keeps Nerfstudio's pinned numpy/torch versions; the worker only needs httpx + sqlalchemy.
!pip -q install --no-deps -e {REPO}
!pip -q install httpx "sqlalchemy>=2"
os.chdir(REPO)
!which ns-train ns-process-data ns-export colmap ffmpeg

## 3. Connect to your laptop

Paste the tunnel URL (looks like `https://something.trycloudflare.com`) and the worker token.
The token is read with `getpass`, so it is not saved in the notebook. If you stored it in
Colab **Secrets** (key icon on the left) as `WORKER_TOKEN`, it is used automatically.

Optional: keep a copy of every trained splat in Google Drive, in case the upload is interrupted.

In [ ]:
import os
from getpass import getpass

os.environ["WORKER_GATEWAY_URL"] = input("Tunnel URL (https://…trycloudflare.com): ").strip().rstrip("/")
try:
    from google.colab import userdata
    os.environ["WORKER_TOKEN"] = userdata.get("WORKER_TOKEN")
    print("Using WORKER_TOKEN from Colab Secrets")
except Exception:
    os.environ["WORKER_TOKEN"] = getpass("WORKER_TOKEN: ").strip()

KEEP_IN_DRIVE = True
if KEEP_IN_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["SPLAT_KEEP_ROOT"] = "/content/drive/MyDrive/indoor_splat"

import httpx
r = httpx.get(os.environ["WORKER_GATEWAY_URL"] + "/worker/ping",
              headers={"Authorization": "Bearer " + os.environ["WORKER_TOKEN"]}, timeout=30)
print("gateway:", r.status_code, r.text)
assert r.status_code == 200, "Check the tunnel URL and that WORKER_TOKEN matches the laptop."

## 4. Run the worker

Leave this cell running. It trains queued splats one after another and waits for new ones.
Progress shows live on the laptop (Processing jobs and the project page). Stop it with the
■ button; the current job is then marked failed and can be trained again.

Settings: `SPLAT_NUM_FRAMES` (frames sampled from the video, default 180) and
`SPLAT_TRAIN_STEPS` (default 30000; 7000 gives a quick preview in about a quarter of the time).

In [ ]:
import os
os.environ.setdefault("SPLAT_NUM_FRAMES", "180")
os.environ.setdefault("SPLAT_TRAIN_STEPS", "30000")
os.environ["SPLAT_WORK_DIR"] = "/content/splat_work"
gpu = __import__("torch").cuda.get_device_name(0).replace(" ", "-")
!python -m queue.workers --pipeline splat --backend splatfacto --name colab-{gpu}

## Troubleshooting

| Symptom | Fix |
|---|---|
| `gateway: 401` | The token differs from `WORKER_TOKEN` on the laptop. |
| `gateway` connection error | The tunnel or `python -m api.worker_gateway` stopped on the laptop; restart it and paste the **new** URL (quick tunnels change URL every start). |
| *Splat failed: … only N MB* | Too few frames registered. Film slower, with more overlap, and train again. |
| *GPU worker stopped responding* | Colab disconnected. Reconnect, rerun cells 1–4, and press **Train again** on the project page. |
| Out of GPU memory | Lower `SPLAT_NUM_FRAMES` (e.g. 120) and rerun cell 4. |